# IMPORT LIBRARIES

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd

In [3]:
train_dir = "/content/drive/MyDrive/dataset/training_set"
test_dir = "/content/drive/MyDrive/dataset/test_set"

img_size = (224, 224)
batch_size = 32

#  IMAGE DATA GENERATOR # This does:
# - rescale pixel values (0–255 → 0–1)
# - data augmentation (rotate, zoom, flip)
# - splits training into train + validation -->

In [4]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,          # normalize images
    validation_split=0.2,    # 20% data for validation
    rotation_range=20,       # random rotation
    zoom_range=0.2,          # random zoom
    horizontal_flip=True     # flip images
)

# Test data: only rescale (no augmentation)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

# Load TRAIN data (80%)

In [5]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)


Found 6410 images belonging to 2 classes.


# Load VALIDATION data (20%)

In [6]:
val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

Found 1601 images belonging to 2 classes.


# Load TEST data

In [7]:
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

Found 2000 images belonging to 2 classes.


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping


# TRANSFER LEARNING MODEL

In [9]:
# Load Pretrained Base Model (ImageNet weights)
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,      # remove original output layer
    weights='imagenet'
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


# Freeze pretrained layers

In [10]:
base_model.trainable = False

In [16]:
model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(128, activation='relu'),

    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

# COMPILE MODEL

In [17]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# EARLY STOPPING

In [13]:
early_stop = EarlyStopping(
    patience=3,
    restore_best_weights=True
)

# TRAIN MODEL

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
 82/201 ━━━━━━━━━━━━━━━━━━━━ 43:49 22s/step - accuracy: 0.9185 - loss: 0.1872

#PLOT RESULTS

In [ ]:
# Accuracy graph

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.title("Accuracy")
plt.show()

# Loss graph

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title("Loss")
plt.show()



#  TEST EVALUATION

In [ ]:
# Check performance on unseen data

loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

#  PREDICTIONS (VISUAL)
class_names = list(train_data.class_indices.keys())

# Get one batch of test images
images, labels = next(test_data)

# Predict
preds = model.predict(images)

# Show first 9 images
plt.figure(figsize=(10,10))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])

    # predicted class
    pred_label = class_names[np.argmax(preds[i])]

    # true class
    true_label = class_names[np.argmax(labels[i])]

    # green = correct, red = wrong
    color = "green" if pred_label == true_label else "red"

    plt.title(f"P:{pred_label}\nT:{true_label}", color=color)
    plt.axis("off")

plt.show()